# SWEC-ETHZ short-term preprocessing

This notebook prepares **window-level features** for the SWEC-ETHZ short-term dataset.

This version restores:

- **quick / full processing**
- clearer configuration
- a **safer CobraBox path** that is more likely to work across versions

## Main idea

- load one seizure file at a time
- optionally crop to a smaller interval for quick tests
- keep the **first 20 channels**
- compute window-level features
- assign `"seizure"` / `"non_seizure"` labels after feature extraction
- save one combined table for later notebooks

## Important note

This notebook prefers **compatibility and clarity** over the most aggressive optimization.

## What to edit first

Most users only need to edit the configuration block below.

### Main switches

- `RUN_MODE = "quick"` or `"full"`
- `DATA_ROOT`
- `WINDOW_SECONDS`
- `OVERLAP_SECONDS`
- `FIRST_N_CHANNELS`

### Quick mode

Quick mode is meant for fast testing.

You can control:
- `QUICK_N_SUBJECTS`
- `QUICK_FILES_PER_SUBJECT`
- `QUICK_PREICTAL_SECONDS`
- `QUICK_ICTAL_SECONDS`

### Full mode

Full mode uses the selected subjects and all available files unless limited.

In [1]:
from pathlib import Path

# ============================================================
# Paths
# ============================================================

DATA_ROOT = Path("../data/remote/swiss_eeg_short")
# Example:
# DATA_ROOT = Path(r"C:\Users\rescic\PycharmProjects\cobrabox\data\remote\swiss_eeg_short")

OUTPUT_DIR = Path("derived_features_swec_short")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Run mode
# ============================================================

RUN_MODE = "full"   # "quick" or "full"

# ============================================================
# Subject selection
# ============================================================

VALID_SUBJECT_IDS = [
    "ID1", "ID2", "ID4a", "ID4b", "ID5", "ID6", "ID7", "ID8", "ID9",
    "ID10", "ID11", "ID12", "ID13a", "ID13b", "ID14a", "ID14b", "ID15", "ID16"
]

SUBJECT_IDS = VALID_SUBJECT_IDS.copy()

# ============================================================
# Quick mode controls
# ============================================================

QUICK_N_SUBJECTS = 6
QUICK_FILES_PER_SUBJECT = 2
QUICK_PREICTAL_SECONDS = 60.0
QUICK_ICTAL_SECONDS = 60.0

# ============================================================
# Full mode controls
# ============================================================

MAX_SEIZURE_FILES_PER_SUBJECT = None   # None = all files

# ============================================================
# Signal settings
# ============================================================

FS = 512.0
FIRST_N_CHANNELS = 20
WINDOW_SECONDS = 2.0
OVERLAP_SECONDS = 0.5

# ============================================================
# Feature settings
# ============================================================

FEATURE_NAMES = ["line_length", "amp_var", "mean", "max", "min", "autocorr","fractal_katz","spike_count"]#"sample_entropy"]

# ============================================================
# Labels
# ============================================================

POSITIVE_LABEL = "seizure"
NEGATIVE_LABEL = "non_seizure"

# ============================================================
# CobraBox mode
# ============================================================

# "safe_apply" = more compatible, usually slower
# "native_stream" = tries Chord + SlidingWindow + ConcatAggregate
COBRABOX_MODE = "safe_apply"

## Imports

In [169]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.io as sio
import xarray as xr

try:
    import cobrabox as cb
except Exception as e:
    raise ImportError(
        "Could not import cobrabox. Please make sure the package is installed "
        "in the notebook environment."
    ) from e

try:
    from IPython.display import display
except Exception:
    display = print

## Subject discovery and run-mode selection

In [170]:
available_subject_ids = [sid for sid in SUBJECT_IDS if (DATA_ROOT / sid).exists()]
missing_subject_ids = [sid for sid in SUBJECT_IDS if not (DATA_ROOT / sid).exists()]

print("DATA_ROOT:", DATA_ROOT.resolve())
print("Requested subjects:", len(SUBJECT_IDS))
print("Available subjects:", len(available_subject_ids))
if missing_subject_ids:
    print("Missing subjects:", missing_subject_ids)

SUBJECT_IDS = available_subject_ids

if RUN_MODE == "quick":
    SUBJECT_IDS = SUBJECT_IDS[:QUICK_N_SUBJECTS]
    effective_max_files = QUICK_FILES_PER_SUBJECT
elif RUN_MODE == "full":
    effective_max_files = MAX_SEIZURE_FILES_PER_SUBJECT
else:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")

print("RUN_MODE:", RUN_MODE)
print("Using SUBJECT_IDS:", SUBJECT_IDS)
print("effective_max_files:", effective_max_files)

DATA_ROOT: C:\Users\rescic\PycharmProjects\cobrabox\data\remote\swiss_eeg_short
Requested subjects: 18
Available subjects: 18
RUN_MODE: full
Using SUBJECT_IDS: ['ID1', 'ID2', 'ID4a', 'ID4b', 'ID5', 'ID6', 'ID7', 'ID8', 'ID9', 'ID10', 'ID11', 'ID12', 'ID13a', 'ID13b', 'ID14a', 'ID14b', 'ID15', 'ID16']
effective_max_files: None


## Helper functions

This notebook uses CobraBox for feature computation, but labels are assigned separately.

To improve compatibility across CobraBox versions, we try a few ways to build the data object:
- `cb.from_numpy(...)`
- `cb.EEG.from_xarray(...)`

If the faster native stream mode does not work in your environment, use:

```python
COBRABOX_MODE = "safe_apply"
```

That mode is more conservative and usually more robust.

In [171]:
def load_swec_shortterm_mat(mat_path, fs=FS):
    mat_path = Path(mat_path)
    d = sio.loadmat(mat_path)
    eeg = d["EEG"]

    if eeg.ndim != 2:
        raise ValueError(f"Expected 2D EEG array in {mat_path}, got shape {eeg.shape}")

    if eeg.shape[1] < FIRST_N_CHANNELS:
        raise ValueError(
            f"{mat_path.name} has only {eeg.shape[1]} channels, "
            f"but FIRST_N_CHANNELS={FIRST_N_CHANNELS}"
        )

    eeg = eeg[:, :FIRST_N_CHANNELS]

    n_samples, n_channels = eeg.shape
    duration_s = n_samples / fs

    seizure_start_s = 180.0
    seizure_end_s = max(seizure_start_s, duration_s - 180.0)

    return {
        "eeg": eeg,
        "fs": fs,
        "n_samples": n_samples,
        "n_channels": n_channels,
        "duration_s": duration_s,
        "seizure_start_s": seizure_start_s,
        "seizure_end_s": seizure_end_s,
        "file_name": mat_path.name,
        "file_stem": mat_path.stem,
        "path": str(mat_path),
    }


def crop_recording(eeg, fs, seizure_start_s, seizure_end_s):
    if RUN_MODE != "quick":
        return eeg, 0.0

    crop_start_s = max(0.0, seizure_start_s - QUICK_PREICTAL_SECONDS)
    crop_end_s = min(len(eeg) / fs, seizure_start_s + QUICK_ICTAL_SECONDS)

    start_idx = int(round(crop_start_s * fs))
    end_idx = int(round(crop_end_s * fs))

    eeg_crop = eeg[start_idx:end_idx]
    return eeg_crop, crop_start_s


def build_cobrabox_data(eeg, fs):
    if hasattr(cb, "from_numpy"):
        try:
            return cb.from_numpy(arr=eeg, dims=["time", "space"], sampling_rate=float(fs))
        except Exception:
            pass

    if hasattr(cb, "EEG") and hasattr(cb.EEG, "from_xarray"):
        da = xr.DataArray(
            eeg,
            dims=["time", "space"],
            coords={
                "time": np.arange(eeg.shape[0]) / fs,
                "space": np.arange(eeg.shape[1]),
            },
        )
        return cb.EEG.from_xarray(da, sampling_rate=float(fs))

    raise RuntimeError(
        "Could not build a CobraBox data object. "
        "Your local CobraBox version may use a different constructor."
    )


def build_cobrabox_feature_map(fs):
    feature_map = {
        "line_length": cb.feature.LineLength(),
        "amp_var": cb.feature.AmplitudeVariation(),
        "mean": cb.feature.Mean(dim="time"),
        "max": cb.feature.Max(dim="time"),
        "min": cb.feature.Min(dim="time"),
        "autocorr": cb.feature.Autocorr(dim="time", fs=fs),
        "fractal_katz": cb.feature.FractalDimKatz(),
        "spike_count": cb.feature.SpikeCount(),
        #"sample_entropy": cb.feature.SampleEntropy(),
    }

    return {k: v for k, v in feature_map.items() if k in FEATURE_NAMES}


def feature_result_to_array(result):
    values = getattr(result, "values", None)
    if values is None and hasattr(result, "data"):
        values = getattr(result.data, "values", None)
    if values is None:
        values = np.asarray(result)
    else:
        values = np.asarray(values)
    return np.squeeze(values)


def build_window_time_table(n_samples, fs, window_seconds, overlap_seconds, offset_s=0.0):
    win = int(round(window_seconds * fs))
    step = int(round((window_seconds - overlap_seconds) * fs))
    if step <= 0:
        raise ValueError("OVERLAP_SECONDS must be smaller than WINDOW_SECONDS")

    starts = np.arange(0, max(0, n_samples - win + 1), step, dtype=int)
    ends = starts + win

    df = pd.DataFrame({
        "window_index": np.arange(len(starts), dtype=int),
        "start_idx": starts,
        "end_idx": ends,
    })
    df["window_start_s"] = offset_s + df["start_idx"] / fs
    df["window_end_s"] = offset_s + df["end_idx"] / fs
    return df


def assign_window_labels(window_df, seizure_start_s, seizure_end_s):
    out = window_df.copy()
    overlaps = (
        (out["window_start_s"] < seizure_end_s) &
        (out["window_end_s"] > seizure_start_s)
    )
    out["target"] = np.where(overlaps, POSITIVE_LABEL, NEGATIVE_LABEL)
    return out

## CobraBox extraction functions

There are two modes:

- `safe_apply`: creates each window explicitly, then applies CobraBox features
- `native_stream`: tries CobraBox `SlidingWindow + Chord + ConcatAggregate`

If `native_stream` fails in your environment, switch back to `safe_apply`.

In [172]:
def extract_features_safe_apply(eeg, fs):
    feature_map = build_cobrabox_feature_map(fs)
    if not feature_map:
        raise ValueError("No features selected in FEATURE_NAMES")

    win = int(round(WINDOW_SECONDS * fs))
    step = int(round((WINDOW_SECONDS - OVERLAP_SECONDS) * fs))
    starts = np.arange(0, max(0, len(eeg) - win + 1), step, dtype=int)

    rows = []
    for start in starts:
        stop = start + win
        window = eeg[start:stop]

        data_obj = build_cobrabox_data(window, fs)
        row = {}

        for feature_name, feature_obj in feature_map.items():
            res = feature_obj.apply(data_obj)
            arr = np.asarray(feature_result_to_array(res))

            if arr.ndim == 0:
                row[f"{feature_name}__value"] = float(arr)

            elif arr.ndim == 1:
                for i, v in enumerate(arr):
                    row[f"{feature_name}__ch{i+1:02d}"] = float(v)

            elif arr.ndim == 2:
                # Keep all values, especially for autocorr
                if arr.shape[0] == window.shape[1]:
                    # channels x lags/features
                    for ch in range(arr.shape[0]):
                        for j in range(arr.shape[1]):
                            row[f"{feature_name}__ch{ch+1:02d}__k{j+1:02d}"] = float(arr[ch, j])
                elif arr.shape[1] == window.shape[1]:
                    # lags/features x channels
                    for j in range(arr.shape[0]):
                        for ch in range(arr.shape[1]):
                            row[f"{feature_name}__ch{ch+1:02d}__k{j+1:02d}"] = float(arr[j, ch])
                else:
                    flat = arr.reshape(-1)
                    for i, v in enumerate(flat):
                        row[f"{feature_name}__f{i+1:03d}"] = float(v)

            else:
                flat = arr.reshape(-1)
                for i, v in enumerate(flat):
                    row[f"{feature_name}__f{i+1:03d}"] = float(v)

        rows.append(row)

    return pd.DataFrame(rows)


def extract_features_native_stream(eeg, fs):
    feature_map = build_cobrabox_feature_map(fs)
    if not feature_map:
        raise ValueError("No features selected in FEATURE_NAMES")

    step_seconds = WINDOW_SECONDS - OVERLAP_SECONDS
    if step_seconds <= 0:
        raise ValueError("OVERLAP_SECONDS must be smaller than WINDOW_SECONDS")

    data_obj = build_cobrabox_data(eeg, fs)
    blocks = []

    for feature_name, feature_obj in feature_map.items():
        chord = cb.Chord(
            split=cb.feature.SlidingWindow(
                window_size=float(WINDOW_SECONDS),
                step_size=float(step_seconds),
            ),
            pipeline=feature_obj,
            aggregate=cb.feature.ConcatAggregate(),
        )
        res = chord.apply(data_obj)
        arr = np.asarray(feature_result_to_array(res))

        if arr.ndim == 0:
            arr = arr.reshape(1, 1)
        elif arr.ndim == 1:
            arr = arr[:, None]
        elif arr.ndim > 2:
            arr = arr.reshape(arr.shape[0], -1)

        # Always preserve all output columns
        if feature_name == "autocorr":
            cols = [f"{feature_name}__f{i+1:03d}" for i in range(arr.shape[1])]
        else:
            cols = [f"{feature_name}__ch{i+1:02d}" for i in range(arr.shape[1])]

        blocks.append(pd.DataFrame(arr, columns=cols))

    return pd.concat(blocks, axis=1)


def extract_feature_table_for_recording(mat_path, subject_id, fs=FS):
    rec = load_swec_shortterm_mat(mat_path, fs=fs)

    eeg_crop, crop_offset_s = crop_recording(
        eeg=rec["eeg"],
        fs=rec["fs"],
        seizure_start_s=rec["seizure_start_s"],
        seizure_end_s=rec["seizure_end_s"],
    )

    if COBRABOX_MODE == "safe_apply":
        feature_df = extract_features_safe_apply(eeg_crop, rec["fs"])
    elif COBRABOX_MODE == "native_stream":
        feature_df = extract_features_native_stream(eeg_crop, rec["fs"])
    else:
        raise ValueError("COBRABOX_MODE must be 'safe_apply' or 'native_stream'")

    time_df = build_window_time_table(
        n_samples=len(eeg_crop),
        fs=rec["fs"],
        window_seconds=WINDOW_SECONDS,
        overlap_seconds=OVERLAP_SECONDS,
        offset_s=crop_offset_s,
    )

    n_rows = min(len(time_df), len(feature_df))
    time_df = time_df.iloc[:n_rows].reset_index(drop=True)
    feature_df = feature_df.iloc[:n_rows].reset_index(drop=True)

    window_df = pd.concat([time_df, feature_df], axis=1)
    window_df = assign_window_labels(
        window_df=window_df,
        seizure_start_s=rec["seizure_start_s"],
        seizure_end_s=rec["seizure_end_s"],
    )

    recording_id = f"{subject_id}__{rec['file_stem']}"
    window_df.insert(0, "subject_id", subject_id)
    window_df.insert(1, "recording_id", recording_id)

    manifest_row = pd.DataFrame([{
        "subject_id": subject_id,
        "recording_id": recording_id,
        "fileName": rec["file_name"],
        "original_n_samples": rec["n_samples"],
        "used_n_samples": len(eeg_crop),
        "n_channels": eeg_crop.shape[1],
        "duration_s": rec["duration_s"],
        "crop_offset_s": crop_offset_s,
        "seizure_start_s": rec["seizure_start_s"],
        "seizure_end_s": rec["seizure_end_s"],
        "window_seconds": WINDOW_SECONDS,
        "overlap_seconds": OVERLAP_SECONDS,
        "n_windows": len(window_df),
        "run_mode": RUN_MODE,
        "cobrabox_mode": COBRABOX_MODE,
    }])

    return window_df, manifest_row

## Quick smoke test on one recording

In [173]:
example_subject = SUBJECT_IDS[0] if SUBJECT_IDS else None
example_files = sorted((DATA_ROOT / example_subject).glob('*.mat')) if example_subject else []

if example_subject is None or not example_files:
    print("No example subject or .mat files found.")
else:
    try:
        df_example, manifest_example = extract_feature_table_for_recording(
            mat_path=example_files[0],
            subject_id=example_subject,
            fs=FS,
        )
        print("Example feature table shape:", df_example.shape)
        display(df_example.head())
        display(manifest_example)
    except Exception as e:
        print("Smoke test failed:")
        print(type(e).__name__, str(e))

Example feature table shape: (323, 149)


,subject_id,recording_id,window_index,start_idx,end_idx,window_start_s,window_end_s,line_length__ch01,line_length__ch02,line_length__ch03,...,fractal_katz__ch13,fractal_katz__ch14,fractal_katz__ch15,fractal_katz__ch16,fractal_katz__ch17,fractal_katz__ch18,fractal_katz__ch19,fractal_katz__ch20,spike_count__value,target
0,ID1,ID1__Sz1,0,0,1024,0.0,2.0,5229.347276,4594.119886,4219.020444,...,1.336396,1.251227,1.296562,1.248318,1.242523,1.316254,1.264996,1.355751,722.0,non_seizure
1,ID1,ID1__Sz1,1,768,1792,1.5,3.5,5055.988465,4123.746468,3819.295340,...,1.328761,1.246394,1.310486,1.245309,1.238862,1.326919,1.252854,1.352484,975.0,non_seizure
2,ID1,ID1__Sz1,2,1536,2560,3.0,5.0,4599.845169,3996.258268,3798.343237,...,1.320818,1.241204,1.323175,1.254246,1.235667,1.326022,1.249129,1.347130,612.0,non_seizure
3,ID1,ID1__Sz1,3,2304,3328,4.5,6.5,4877.313455,4237.723782,3825.288739,...,1.306409,1.252840,1.341537,1.273760,1.263780,1.327308,1.265760,1.353951,676.0,non_seizure
4,ID1,ID1__Sz1,4,3072,4096,6.0,8.0,4454.805991,3902.546177,3584.420309,...,1.313322,1.250134,1.339874,1.244400,1.256925,1.338257,1.248757,1.351436,1163.0,non_seizure


,subject_id,recording_id,fileName,original_n_samples,used_n_samples,n_channels,duration_s,crop_offset_s,seizure_start_s,seizure_end_s,window_seconds,overlap_seconds,n_windows,run_mode,cobrabox_mode
0,ID1,ID1__Sz1,Sz1.mat,248320,248320,20,485.0,0.0,180.0,305.0,2.0,0.5,323,full,safe_apply


## Run extraction over all selected subjects

If this is slow, reduce:
- `QUICK_N_SUBJECTS`
- `QUICK_FILES_PER_SUBJECT`
- `FEATURE_NAMES`

If `native_stream` fails, switch:
```python
COBRABOX_MODE = "safe_apply"
```

In [174]:
subject_tables = []
manifest_rows = []
skipped_rows = []

for subject_id in SUBJECT_IDS:
    subject_dir = DATA_ROOT / subject_id
    mat_files = sorted(subject_dir.glob("*.mat"))

    if effective_max_files is not None:
        mat_files = mat_files[:effective_max_files]

    print(f"Subject {subject_id}: {len(mat_files)} file(s)")

    for mat_path in mat_files:
        try:
            df_rec, df_manifest = extract_feature_table_for_recording(
                mat_path=mat_path,
                subject_id=subject_id,
                fs=FS,
            )
            subject_tables.append(df_rec)
            manifest_rows.append(df_manifest)
        except Exception as e:
            print(f"  Skipping {mat_path.name}: {type(e).__name__}: {e}")
            skipped_rows.append({
                "subject_id": subject_id,
                "fileName": mat_path.name,
                "reason": f"{type(e).__name__}: {e}",
            })

if not subject_tables:
    raise RuntimeError("No recordings were successfully processed.")

all_window_features = pd.concat(subject_tables, ignore_index=True)
recording_manifest = pd.concat(manifest_rows, ignore_index=True)
skipped_df = pd.DataFrame(skipped_rows)

print("All window features shape:", all_window_features.shape)
print("Recording manifest shape:", recording_manifest.shape)
if not skipped_df.empty:
    print("Skipped files:", len(skipped_df))

Subject ID1: 13 file(s)
Subject ID2: 4 file(s)
Subject ID4a: 7 file(s)
Subject ID4b: 7 file(s)
Subject ID5: 10 file(s)
Subject ID6: 4 file(s)
Subject ID7: 2 file(s)
Subject ID8: 2 file(s)
Subject ID9: 9 file(s)
Subject ID10: 5 file(s)
Subject ID11: 2 file(s)
Subject ID12: 10 file(s)
Subject ID13a: 4 file(s)
Subject ID13b: 3 file(s)
Subject ID14a: 4 file(s)
Subject ID14b: 3 file(s)
Subject ID15: 3 file(s)
Subject ID16: 6 file(s)
All window features shape: (31774, 149)
Recording manifest shape: (98, 15)


## Dataset summary tables

In [175]:
participant_summary = (
    recording_manifest.groupby("subject_id")
    .agg(
        n_recordings=("recording_id", "nunique"),
        mean_duration_s=("duration_s", "mean"),
        mean_channels=("n_channels", "mean"),
        mean_windows=("n_windows", "mean"),
    )
    .reset_index()
)

window_balance = (
    all_window_features.groupby(["subject_id", "target"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

participant_summary = participant_summary.merge(window_balance, on="subject_id", how="left")

overall_summary = pd.DataFrame([{
    "n_subjects": all_window_features["subject_id"].nunique(),
    "n_recordings": all_window_features["recording_id"].nunique(),
    "n_windows": len(all_window_features),
    "n_features": len([c for c in all_window_features.columns if "__ch" in c]),
    "window_seconds": WINDOW_SECONDS,
    "overlap_seconds": OVERLAP_SECONDS,
    "first_n_channels": FIRST_N_CHANNELS,
    "run_mode": RUN_MODE,
    "cobrabox_mode": COBRABOX_MODE,
}])

display(participant_summary)
display(overall_summary)

,subject_id,n_recordings,mean_duration_s,mean_channels,mean_windows,non_seizure,seizure
0,ID1,13,431.076923,20.0,286.692308,3094,633
1,ID10,5,374.400000,20.0,249.000000,1190,55
2,ID11,2,469.000000,20.0,312.000000,476,148
3,ID12,10,404.800000,20.0,269.200000,2380,312
4,ID13a,4,432.500000,20.0,287.500000,952,198
5,ID13b,3,449.333333,20.0,299.000000,714,183
6,ID14a,4,853.500000,20.0,568.250000,952,1321
7,ID14b,3,1070.666667,20.0,713.000000,714,1425
8,ID15,3,480.666667,20.0,319.666667,714,245
9,ID16,6,448.000000,20.0,298.000000,1428,360


,n_subjects,n_recordings,n_windows,n_features,window_seconds,overlap_seconds,first_n_channels,run_mode,cobrabox_mode
0,18,98,31774,140,2.0,0.5,20,full,safe_apply


## Save outputs

In [176]:
all_window_features.to_csv(OUTPUT_DIR / "all_window_features.csv", index=False)
recording_manifest.to_csv(OUTPUT_DIR / "recording_manifest.csv", index=False)
participant_summary.to_csv(OUTPUT_DIR / "participant_summary.csv", index=False)
overall_summary.to_csv(OUTPUT_DIR / "overall_summary.csv", index=False)
if not skipped_df.empty:
    skipped_df.to_csv(OUTPUT_DIR / "skipped_files.csv", index=False)

try:
    all_window_features.to_parquet(OUTPUT_DIR / "all_window_features.parquet", index=False)
    print("Saved parquet feature table.")
except Exception as e:
    print("Could not save parquet file:", e)

print("Saved outputs to:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

Could not save parquet file: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
Saved outputs to: C:\Users\rescic\PycharmProjects\cobrabox\examples\derived_features_swec_short
- all_window_features.csv
- overall_summary.csv
- participant_summary.csv
- recording_manifest.csv
